# Phase currents in rings, meshes and parallel cables

In a line every branch between the infeed and the fault carries the full
fault current. As soon as the network has a second route — a ring, a mesh,
or simply a second cable between the same two stations — the current
divides, and how it divides decides how hard each cable shield is driven.

`groundinsight` finds the split by solving a reduced network of the phase
conductors alone, with the fault bus as reference
(`phase_current_mode="auto"`, the default). The legacy mode
`phase_current_mode="paths"` gives every branch on any source-to-fault path
the *full* source current, scaled by its `parallel_coefficient`; it is kept
to reproduce older studies and for a split known from measurement. This
notebook

1. compares both modes on a line, a ring, two parallel cables and a mesh,
2. shows that `BranchType.phase_impedance_formula` decides the split, and
3. checks every result against a **fully coupled multi-conductor solve**
   written here in NumPy: phase conductors, shields and electrodes in one
   system, without the one-way coupling that `groundinsight` uses.

Background: [Phase currents in rings and meshes](../../explanation/phase-currents/).

In [ ]:
import warnings

import numpy as np
import polars as pl

import groundinsight as gi

pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(60)
print(f"groundinsight {gi.__version__}")

## The cable and the stations

One illustrative cable for every branch, 1 km per branch, at 50 Hz:

| per km | value | loop |
|---|---|---|
| $Z_s'$ | $0.1 + 0.2j\ \Omega$ | shield with earth return |
| $Z_m'$ | $0.05 + 0.1j\ \Omega$ | coupling between the phase conductor and that loop |
| $Z_{pp}'$ | $0.2 + 0.7j\ \Omega$ | phase conductor with earth return, on its own |

The ratio $k = Z_m/Z_s$ is exactly $0.5$, which makes the difference between
the two modes as plain as it gets. `phase_impedance_formula` takes the
phase–earth loop *with the shield bonded at both ends*,
$Z_\text{red} = Z_{pp} - Z_m^2/Z_s$. Every station has a $10\ \Omega$
electrode; the source feeds 1000 A at `S`, the fault is at `F`.

In [ ]:
FREQUENCY = 50.0
I_SOURCE = 1000.0          # A
Z_ELECTRODE = 10.0         # Ohm, every station

ZS = 0.1 + 0.2j            # Ohm/km  shield - earth loop
ZM = 0.05 + 0.1j           # Ohm/km  phase conductor <-> shield loop
ZPP = 0.2 + 0.7j           # Ohm/km  phase conductor - earth loop on its own
Z_RED = ZPP - ZM**2 / ZS   # Ohm/km  phase - earth loop, shield bonded at both ends

print(f"k = Z_m/Z_s = {ZM / ZS}")
print(f"Z_red'      = {Z_RED:.4f} Ohm/km")


def per_km(z):
    "A constant per-km impedance as a formula in the length l."
    return f"({z.real!r} + {z.imag!r}*j)*l"


STATION = gi.BusType(name="station", system_type="MV", voltage_level=20.0,
                     impedance_formula=repr(Z_ELECTRODE))


def cable(name="cable", z_red=Z_RED):
    "The cable type; ``z_red=None`` leaves the phase impedance undeclared."
    return gi.BranchType(
        name=name, grounding_conductor=True,
        self_impedance_formula=per_km(ZS),
        mutual_impedance_formula=per_km(ZM),
        phase_impedance_formula=None if z_red is None else per_km(z_red),
    )


def build(branches, *, types=None, parallel_coefficient=1.0, default=None):
    "A network of 1 km branches ``(name, from_bus, to_bus)``, source at S, fault at F."
    default = default or cable()
    net = gi.create_network(name="mesh", frequencies=[FREQUENCY])
    for bus in sorted({bus for _, a, b in branches for bus in (a, b)}):
        gi.create_bus(name=bus, type=STATION, network=net)
    for name, a, b in branches:
        gi.create_branch(name=name, type=(types or {}).get(name, default), from_bus=a,
                         to_bus=b, length=1.0, parallel_coefficient=parallel_coefficient,
                         network=net)
    gi.create_source(name="infeed", bus="S", values={FREQUENCY: I_SOURCE}, network=net)
    gi.create_fault(name="F", bus="F", network=net)
    return net


def solve(net, mode="auto"):
    "Run the fault; the 50 Hz row of res_fault and the diagnostic codes of the run."
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")   # every warning is kept in result.diagnostics
        result = gi.run_fault(net, fault="F", phase_current_mode=mode)
    return net.res_fault("F").row(0, named=True), sorted({d.code for d in result.diagnostics})

## The reference: everything coupled, solved at once

The reference does not split the phase current first. Every branch carries
two unknown currents, $I_p$ in the phase conductor and $I_s$ in the shield
(both from `from_bus` to `to_bus`), and obeys the two loop equations with
earth return

$$
u_p(a) - u_p(b) = Z_{pp}\,I_p + Z_m\,I_s, \qquad
u_e(a) - u_e(b) = Z_m\,I_p + Z_s\,I_s ,
$$

where $u_p$ is the potential of the phase conductor and $u_e$ that of the
earthing system at a bus. The source drives $I$ into the phase conductor at
`S` and draws it from the earthing system at `S`; the fault ties the phase
conductor to the earthing system at `F`. That is the physical sign frame, in
which the earth potentials of `groundinsight` carry the opposite sign
([Signs and phase angles](../../reference/conventions/#signs-and-phase-angles));
its shield currents are physical from `from_bus` to `to_bus` in both.

In [ ]:
def coupled_reference(branches, zpp=None):
    '''Phase conductors, shields and electrodes in one linear system.

    ``branches`` as for ``build``; ``zpp`` maps a branch name to its Z_pp'
    (default ``ZPP``). Returns earth potentials (physical frame), phase
    currents and shield currents.
    '''
    zpp = zpp or {}
    buses = sorted({bus for _, a, b in branches for bus in (a, b)})
    n, m = len(buses), len(branches)
    ue = {bus: i for i, bus in enumerate(buses)}          # unknown index of u_e
    up = {bus: n + i for i, bus in enumerate(buses)}      # ... of u_p
    ip = {name: 2 * n + k for k, (name, _, _) in enumerate(branches)}
    is_ = {name: 2 * n + m + k for k, (name, _, _) in enumerate(branches)}

    A = np.zeros((2 * n + 2 * m, 2 * n + 2 * m), dtype=complex)
    rhs = np.zeros(2 * n + 2 * m, dtype=complex)
    row = 0
    for name, a, b in branches:                           # two loop laws per branch
        A[row, [up[a], up[b], ip[name], is_[name]]] = 1, -1, -zpp.get(name, ZPP), -ZM
        A[row + 1, [ue[a], ue[b], ip[name], is_[name]]] = 1, -1, -ZM, -ZS
        row += 2
    for bus in buses:                                     # Kirchhoff's current law
        if bus == "F":
            continue
        for name, a, b in branches:
            sign = 1 if a == bus else -1 if b == bus else 0
            A[row, is_[name]] += sign
            A[row + 1, ip[name]] += sign
        A[row, ue[bus]] += 1 / Z_ELECTRODE
        rhs[row], rhs[row + 1] = (-I_SOURCE, I_SOURCE) if bus == "S" else (0, 0)
        row += 2
    A[row, [up["F"], ue["F"]]] = 1, -1                    # the fault: u_p(F) = u_e(F)
    for name, a, b in branches:                           # one node at F
        sign = 1 if a == "F" else -1 if b == "F" else 0
        A[row + 1, [ip[name], is_[name]]] += sign
    A[row + 1, ue["F"]] += 1 / Z_ELECTRODE

    x = np.linalg.solve(A, rhs)
    return ({bus: x[i] for bus, i in ue.items()},
            {name: x[i] for name, i in ip.items()},
            {name: x[i] for name, i in is_.items()})


def phasors(net):
    "Earth potentials and shield currents of a solved network as complex numbers."
    buses = net.res_buses("F").filter(~pl.col("is_rms"))
    branches = net.res_branches("F").filter(~pl.col("is_rms"))
    u = {r["bus_name"]: r["EPR_V"] * np.exp(1j * np.radians(r["EPR_degree"]))
         for r in buses.iter_rows(named=True)}
    i = {r["branch_name"]: r["I_branch_A"] * np.exp(1j * np.radians(r["I_branch_degree"]))
         for r in branches.iter_rows(named=True)}
    return u, i


def deviation(net, reference):
    '''Largest deviation of the earth potentials and of the shield currents from
    the reference, each relative to the largest value of the reference.'''
    u_ref, _, i_ref = reference
    u, i = phasors(net)
    du = max(abs(u[bus] + u_ref[bus]) for bus in u) / max(abs(v) for v in u_ref.values())
    di = max(abs(i[name] - i_ref[name]) for name in i) / max(abs(v) for v in i_ref.values())
    return max(du, di)

## 1  The line: nothing to divide

`S - A - F`, two cable sections. There is one route, it carries everything,
and both modes must agree.

In [ ]:
LINE = [("SA", "S", "A"), ("AF", "A", "F")]


def table(cases):
    "One row per (label, network, mode)."
    rows = []
    for label, net, mode in cases:
        key, codes = solve(net, mode)
        rows.append({"case": label, "U_E_V": key["U_E_V"], "r_current": key["r_current"],
                     "r_epr": key["r_epr"], "Z_E_Ohm": key["Z_E_Ohm"],
                     "diagnostics": ", ".join(codes) or "-"})
    return pl.DataFrame(rows)


line_auto, line_paths = build(LINE), build(LINE)
print(table([("auto", line_auto, "auto"), ("paths", line_paths, "paths")]))

reference = coupled_reference(LINE)
print(f"\ncoupled reference: U_E = {abs(reference[0]['F']):.4f} V")
print(f"largest deviation from the reference: auto {deviation(line_auto, reference):.1e}, "
      f"paths {deviation(line_paths, reference):.1e}")

Identical, and identical to the fully coupled solve: in a line the one-way
coupling is exact, whatever the phase impedance.

## 2  The ring: the current divides

`S - A - F` and `S - B - F`, two routes of two sections each.

In [ ]:
RING = [("SA", "S", "A"), ("AF", "A", "F"), ("SB", "S", "B"), ("BF", "B", "F")]

ring = {
    "auto": build(RING),
    "auto, coefficient 0.5": build(RING, parallel_coefficient=0.5),
    "paths, coefficient 1.0": build(RING),
    "paths, coefficient 0.5": build(RING, parallel_coefficient=0.5),
}
print(table([(label, net, label.split(",")[0]) for label, net in ring.items()]))

reference = coupled_reference(RING)
print(f"\ncoupled reference: U_E = {abs(reference[0]['F']):.4f} V")
print("phase currents of the reference / A:",
      {name: round(float(abs(i)), 3) for name, i in reference[1].items()})
for label, net in ring.items():
    largest = max(row["EPR_V"] for row in net.res_buses("F").filter(~pl.col("is_rms")).iter_rows(named=True))
    print(f"{label:24s} largest EPR in the network {largest:9.4f} V,"
          f" deviation from the reference {deviation(net, reference):.1e}")

`auto` divides the 1000 A into 500 A per route, as the reference does, and
agrees with it to rounding. The legacy mode with the default coefficient 1.0
gives both routes the full 1000 A, so the coupling is counted twice: the
earthing network sees $(1 - 2k)\,I$ instead of $(1 - k)\,I$, and with
$k = 0.5$ that is **zero — every potential in the network vanishes**. The
run warns with `PARALLEL_COEFFICIENT_DEFAULT`: several paths, default
coefficients. A coefficient of 0.5, set by
hand, repairs this ring; `auto` needs nothing and ignores the coefficient.

For other cables the same mistake does not vanish but distorts: the EPR is
scaled by $\lvert 1-2k\rvert / \lvert 1-k\rvert$, which can be larger or smaller
than one ([Enumerated paths and the legacy mode](../../explanation/phase-currents/#path-finding)).

## 3  Two cables between the same stations

The same mechanism without a visible ring: a second cable laid next to the
first, the most common mesh in practice.

In [ ]:
PARALLEL = [("C1", "S", "F"), ("C2", "S", "F")]
SINGLE = [("C1", "S", "F")]

two, two_paths, one = build(PARALLEL), build(PARALLEL), build(SINGLE)
print(table([("two cables, auto", two, "auto"),
             ("two cables, paths", two_paths, "paths"),
             ("one cable, auto", one, "auto")]))

for label, net, branches in (("two cables", two, PARALLEL), ("one cable", one, SINGLE)):
    reference = coupled_reference(branches)
    print(f"{label}: coupled reference U_E = {abs(reference[0]['F']):.4f} V,"
          f" deviation {deviation(net, reference):.1e}")

With two cables each one carries half of the phase current and the shields
are in parallel: the EPR falls to about half that of one cable, as the
reference confirms. The legacy mode collapses to zero again — two parallel
cables are a ring of two branches.

## 4  A mesh, built in two orders

Five branches between four stations: `S - A`, `A - B`, `B - F`, `S - B`,
`A - F`. The answer must not depend on the order in which the branches were
added to the network.

In [ ]:
MESH = [("SA", "S", "A"), ("AB", "A", "B"), ("BF", "B", "F"), ("SB", "S", "B"), ("AF", "A", "F")]
reference = coupled_reference(MESH)
print(f"coupled reference: U_E = {abs(reference[0]['F']):.6f} V")
print("phase currents of the reference / A:",
      {name: round(float(abs(i)), 3) for name, i in reference[1].items()})

rows = []
for mode in ("auto", "paths"):
    for order, branches in (("forward", MESH), ("backward", list(reversed(MESH)))):
        net = build(branches)
        key, codes = solve(net, mode)
        u, i = phasors(net)
        rows.append({"mode": mode, "order": order, "U_E_V": round(key["U_E_V"], 6),
                     "EPR(A)": (f"{abs(u['A']):6.2f} V at {np.degrees(np.angle(u['A'])):7.2f}"
                                if abs(u["A"]) > 1e-9 else "0"),
                     "I_shield(S-A)": f"{abs(i['SA']):6.1f} A",
                     "I_shield(S-B)": f"{abs(i['SB']):6.1f} A",
                     "deviation": f"{deviation(net, reference):.1e}"})
pl.DataFrame(rows)

The mesh is symmetric: in the reference the cross branch `A - B` carries no
phase current, `A` and `B` stay at zero potential, and `S - A` and `S - B`
carry the same shield current. `auto` reproduces that in either order.

The legacy mode gives every branch on any of the four source-to-fault paths
the full 1000 A, the cross branch included, and the direction of that current
in `A - B` is fixed by the path met first — so it depends on the order in
which the branches were added. Stations `A` and `B` rise to 27.88 V, the
shield currents of `S - A` and `S - B` swap with the order, and the angle of
the potential at `A` turns by 180°.

## 5  The phase impedance decides the split

So far every branch was the same cable, and any impedance proportional to the
length would have divided the current correctly. Now route 2 (`S - B - F`)
gets a phase conductor whose $Z_\text{red}$ is ten times that of route 1, at
the same angle; the shields stay the same. The current divider expects
$10/11$ of the current on route 1 and $1/11$ on route 2.

In [ ]:
Z_RED_2 = 10 * Z_RED
weak = cable("weak_phase", z_red=Z_RED_2)
TYPES = {"SB": weak, "BF": weak}

declared = build(RING, types=TYPES)
key, codes = solve(declared)

# The reference needs Z_pp of the second conductor: Z_red + Z_m^2/Z_s.
zpp_2 = Z_RED_2 + ZM**2 / ZS
reference = coupled_reference(RING, zpp={"SB": zpp_2, "BF": zpp_2})

divider = I_SOURCE * (2 * Z_RED_2) / (2 * Z_RED + 2 * Z_RED_2)
print(f"current divider:   route 1 {abs(divider):8.3f} A, route 2 {abs(I_SOURCE - divider):8.3f} A")
print(f"coupled reference: route 1 {abs(reference[1]['SA']):8.3f} A, route 2 {abs(reference[1]['SB']):8.3f} A")
print()
print(f"groundinsight U_E = {key['U_E_V']:.4f} V, reference {abs(reference[0]['F']):.4f} V,"
      f" deviation {deviation(declared, reference):.1e}, diagnostics: {codes or '-'}")

The fully coupled solve divides the phase current exactly as the current
divider does — the one-way coupling is exact here because $k = Z_m/Z_s$ is
the same on every branch of the ring — and `groundinsight` reproduces all
potentials and shield currents of the reference.

The earth potential rise has not moved: 55.6208 V, as in section 2. That is
no accident. A phase current $I_p$ drives the shield of its branch like a
current source $k\,I_p$ between the two ends, and when $k$ is the same on
every branch these sources add up, station by station, to $k$ times what the
phase conductors take in and give off — $kI$ at `S`, $-kI$ at `F`, nothing
in between — however the current divides on the way. With a uniform $k$ the
split moves the **shield currents** and leaves every earth potential where
it is. Without `phase_impedance_formula` the split is a guess: the proxy is
the shield impedance, the same on both routes, so it divides 500 : 500:

In [ ]:
undeclared = build(RING, types={"SB": cable("weak_phase_undeclared", z_red=None),
                                "BF": cable("weak_phase_undeclared", z_red=None)},
                   default=cable("cable_undeclared", z_red=None))
key_proxy, codes = solve(undeclared)
_, i_declared = phasors(declared)
_, i_proxy = phasors(undeclared)

print(f"U_E: declared {key['U_E_V']:.4f} V, proxy {key_proxy['U_E_V']:.4f} V   (proxy run: {', '.join(codes)})")
pl.DataFrame([
    {"branch": name, "reference_A": abs(reference[2][name]),
     "declared_A": abs(i_declared[name]), "proxy_A": abs(i_proxy[name])}
    for name, _, _ in RING
])

The declared formula reproduces the shield currents of the reference: 703 A
in the shields of route 1, 294 A in those of route 2. The proxy puts 499 A
into every shield — the potentials are right, the thermal load of the shields
is not.

In a mixed ring — a cable route and an overhead-line route, with different
$k$ — the split moves the earth potentials as well, and the proxy can be far
off ([One-way coupling: when the split is exact](../../explanation/phase-currents/#one-way-coupling-when-the-split-is-exact)).

## 6  The proxy is announced only where it matters

In a line the proxy cannot change anything — there is one route. The warning
`PHASE_PROXY` therefore stays silent there and appears once per run, naming
the branches, as soon as the phase conductors form a loop around the fault.

In [ ]:
for label, branches in (("line", LINE), ("ring", RING)):
    net = build(branches, default=cable("no_phase_formula", z_red=None))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        result = gi.run_fault(net, fault="F")
    print(f"{label}: {len(caught)} warning(s)")
    for diagnostic in result.diagnostics:
        print(f"  {diagnostic.category} {diagnostic.code}: {diagnostic.elements}")
        print(f"  {diagnostic.message[:150]} ...")

## Summary

| network | `paths`, coefficient 1.0 | `auto` | fully coupled reference |
|---|---|---|---|
| line | the same as `auto` | exact | agrees to rounding |
| ring | every potential zero | 500 A per route | agrees to rounding |
| two parallel cables | every potential zero | 500 A per cable | agrees to rounding |
| mesh | 27.88 V at stations that stay at zero, depends on the order | the same in either order | agrees to rounding |
| ring with two phase conductors | — | 909 A : 91 A, as the phase impedances; without the formula 499 A in every shield instead of 703 A and 294 A | agrees to rounding |

`Branch.parallel_coefficient` is ignored in `auto` and still applies in
`paths`.